In [19]:
import pandas as pd

from ee559_project.hate_baseline import ExperimentConfig
from ee559_project.hate_baseline import evaluate_prediction_csv_overall_and_per_dataset
from ee559_project.hate_baseline import run_experiment

In [10]:
config = ExperimentConfig(
    epochs=3,
    batch_size=16,
    max_length=128,
    learning_rate=2e-5,
    pooling="auto",   # swap to "cls" or "mean"
    head="linear",    # swap to "mlp"
    device="auto",
    datasets=["ETHOS", "HateXplain"],
)

model_names = [
    "bert-base-uncased",
    "roberta-base",
    "microsoft/deberta-v3-base",
]

In [11]:
# CPU verification override: keep run short while testing full pipeline.
config.epochs = 1
config.device = "cpu"
config.max_length = 64
config.batch_size = 8
config.max_train_samples = 64
config.max_val_samples = 32
config.max_test_samples = 64
model_names = ["bert-base-uncased"]
config

ExperimentConfig(train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42, max_length=64, batch_size=8, epochs=1, learning_rate=2e-05, weight_decay=0.01, pooling='auto', head='linear', hidden_dim=256, dropout=0.1, device='cpu', datasets=['ETHOS', 'HateXplain'], max_train_samples=64, max_val_samples=32, max_test_samples=64)

In [12]:
run_summaries = []
for model_name in model_names:
    summary = run_experiment(
        model_name=model_name,
        datasets_dir="datasets",
        output_dir="outputs",
        config=config,
    )
    run_summaries.append(
        {
            "model": summary["model_name"],
            "predictions": summary["prediction_path"],
            "train_val_test": summary["split_sizes"],
            "split_report": summary["split_dataset_report"],
        }
    )

run_results = pd.DataFrame(run_summaries)
run_results

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model,predictions,train_val_test,split_report
0,bert-base-uncased,outputs/predictions_bert-base-uncased.csv,"{'train': 64, 'val': 32, 'test': 64}","[{'dataset': 'ETHOS', 'train': 2, 'val': 4, 't..."


In [13]:
split_results = pd.DataFrame(run_summaries[0]["split_report"])
split_results

,dataset,train,val,test,total
0,ETHOS,2,4,1,7
1,HateXplain,62,28,63,153


In [14]:
evaluation_tables = evaluate_prediction_csv_overall_and_per_dataset(run_summaries[0]["predictions"])
overall_evaluation = evaluation_tables["overall"]
per_dataset_evaluation = evaluation_tables["per_dataset"]
overall_evaluation

,sample_count,accuracy,macro_f1,hateful_f1,non_hate_f1,non_hate_precision,non_hate_recall,hateful_precision,hateful_recall,tn,fp,fn,tp
0,64,0.65625,0.633333,0.725,0.541667,0.541667,0.541667,0.725,0.725,13,11,11,29


In [15]:
per_dataset_evaluation

,dataset,sample_count,accuracy,macro_f1,hateful_f1,non_hate_f1,non_hate_precision,non_hate_recall,hateful_precision,hateful_recall,tn,fp,fn,tp
0,HateXplain,63,0.666667,0.643684,0.734177,0.553191,0.541667,0.565217,0.74359,0.725,13,10,11,29
1,ETHOS,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000,0,1,0,0
